In [19]:
from langchain.tools import tool
import sys
sys.path.append("../..")
from common import init_llm
import json


In [ ]:
@tool
def get_employee_info(employee_id:str)->str:
    """
    根据员工ID去查询员工信息
    Args:
        employee_id (str):员工ID
    Returns:
        str:员工信息
    """
    # 模拟数据
    mock_employee_database = {
        "E001": {"name": "张三", "department": "技术部", "position": "高级软件工程师", "email": "zhangsan@company.com"},
        "E002": {"name": "李四", "department": "市场部", "position": "市场经理", "email": "lisi@company.com"},
        "E003": {"name": "王五", "department": "人力资源部", "position": "招聘专员", "email": "wangwu@company.com"}
    }

    employee_record = mock_employee_database.get(employee_id)
    if employee_record:
        return f"员工ID为{employee_id}的信息{employee_record}"
    else:
        return f"员工ID为{employee_id}的员工不存在"

from pydantic import BaseModel,Field,field_validator
from typing import Optional,Literal # 可选类型，限制参数只能取一部分值
class QueryTicketsSchema(BaseModel):
    ticket_id: Optional[str] = Field(default=None, description="工单ID")
    assigner: Optional[str] = Field(default=None, description="工单分配人")
    status: Optional[Literal["open", "resolved", "closed"]] = Field(default=None, description="工单状态,open(待处理),resolved(已处理),closed(已关闭)")
    priority: Optional[Literal["low", "medium", "high"]] = Field(default=None, description="工单优先级,low(低),medium(中),high(高)")

    @field_validator("ticket_id") # 自定义校验器
    def validate_ticket_id(cls, v):
        return v.upper() if v else None

@tool(args_schema=QueryTicketsSchema) # 利用pydantics为工具的参数做校验
def query_tikets(
    ticket_id:str=None,
    assigner:str=None,
    status:str=None,
    priority:str=None
)->str:
    """
    根据工单ID查询工单的详细信息。
    Args:
        ticket_id (str, optional): 工单ID
        assigner (str, optional): 工单分配人
        status (str, optional): 工单状态
        priority (str, optional): 工单优先级
    Returns:
        str: 工单详细信息
    """
    mock_tickets_db = [
        {"ticket_id": "TK2025012001", "assigner": "张三", "title": "登录页面加载缓慢", "status": "open","priority": "low"},
        {"ticket_id": "TK2025012002", "assigner": "李四", "title": "用户头像上传失败", "status": "open","priority": "medium"},
        {"ticket_id": "TK2025011901", "assigner": "张三", "title": "支付成功通知未发送", "status": "resolved","priority": "high"},
        {"ticket_id": "TK2025011902", "assigner": "马六", "title": "订单查询接口返回空值", "status": "closed","priority": "high"},
    ]

    filtered_tickets = mock_tickets_db
    # 传进来什么参数就用什么
    if ticket_id:
        filtered_tickets = [ticket for ticket in filtered_tickets if ticket["ticket_id"] == ticket_id]
    if assigner:
        filtered_tickets = [ticket for ticket in filtered_tickets if ticket["assigner"] == assigner]
    if status:
        filtered_tickets = [ticket for ticket in filtered_tickets if ticket["status"] == status]
    if priority:
        filtered_tickets = [ticket for ticket in filtered_tickets if ticket["priority"] == priority]

    if not filtered_tickets:
        return "没有找到任何工单"

    result = {
        "total_count":len(filtered_tickets),
        "tickets":filtered_tickets
    }

    return json.dumps(result, ensure_ascii=False, indent=2)

In [22]:
from langchain.agents import create_agent

agent = create_agent(
    model = init_llm.deepseek_llm,
    tools=[get_employee_info,query_tikets],
    system_prompt="你是一个工单查询助手，能够根据工单ID查询详细信息"
)

In [24]:
res = agent.invoke({
    "messages":"请帮我查询一下张三负责的优先级别为高的工单信息"
})
for msg in res["messages"]:
    msg.pretty_print()

================================ Human Message =================================

请帮我查询一下张三负责的优先级别为高的工单信息
================================== Ai Message ==================================

好的，我来查询张三负责的优先级为高的工单信息。
Tool Calls:
  query_tikets (019fc2aa77c02173dfe88290191a8cba)
 Call ID: 019fc2aa77c02173dfe88290191a8cba
  Args:
    assigner: 张三
    priority: high
================================= Tool Message =================================
Name: query_tikets

{
  "total_count": 1,
  "tickets": [
    {
      "ticket_id": "TK2025011901",
      "assigner": "张三",
      "title": "支付成功通知未发送",
      "status": "resolved",
      "priority": "high"
    }
  ]
}
================================== Ai Message ==================================

查询结果如下：

**张三负责的高优先级工单信息：**

| 工单ID | 标题 | 分配人 | 状态 | 优先级 |
|--------|------|--------|------|--------|
| TK2025011901 | 支付成功通知未发送 | 张三 | 已处理（resolved） | 高（high） |

目前张三名下只有 **1 个高优先级工单**，该工单状态为 **已处理**。如果您需要查看更详细的内容，我可以进一步查询该工单的详细信息。
